# Tool Use 기초 (2) — 메시지 블록과 결과 전달

**Skilljar Lessons 05-06 대응**

이 노트북에서 다루는 내용:
1. Tool Use 응답의 content 블록 구조
2. 블록 순회 및 도구 정보 추출
3. `tool_result` 메시지로 결과 전달
4. 최종 텍스트 응답 수신
5. 에러 핸들링 (`is_error`)

In [ ]:
# ── Setup ──────────────────────────────────────────────
import anthropic
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"


# ── Tool function (Notebook 01에서 가져옴) ──
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    valid_formats = [
        "%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%H:%M:%S",
        "%H:%M", "%Y/%m/%d", "%m/%d/%Y",
    ]
    if date_format not in valid_formats:
        raise ValueError(
            f"Invalid date format: {date_format}. Valid formats: {valid_formats}"
        )
    return datetime.now().strftime(date_format)


# ── Tool schema ──
get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time in a specified format. "
        "Defaults to '%Y-%m-%d %H:%M:%S'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "The format string for the date/time output. "
                    "Supported: '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%H:%M:%S', "
                    "'%H:%M', '%Y/%m/%d', '%m/%d/%Y'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

tools = [get_current_datetime_schema]

## §1. Tool Use API 호출 (Making a Tool-Enabled Call)

Claude에게 도구를 제공하고 질문하면, 응답의 `content`에 여러 **블록(block)**이 들어옵니다.

In [ ]:
messages = [{"role": "user", "content": "What time is it right now?"}]

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("stop_reason:", response.stop_reason)
print("content blocks:", len(response.content))
print()
for i, block in enumerate(response.content):
    print(f"Block {i}: {block}")
    print()

## §2. 블록 순회 (Iterating Content Blocks)

`response.content`는 리스트이며, 각 블록은 `type` 속성을 가집니다:
- `"text"` — Claude의 텍스트 응답 (TextBlock)
- `"tool_use"` — 도구 호출 요청 (ToolUseBlock)

In [ ]:
for block in response.content:
    if block.type == "text":
        print(f"[TextBlock] {block.text}")
    elif block.type == "tool_use":
        print(f"[ToolUseBlock]")
        print(f"  id:    {block.id}")
        print(f"  name:  {block.name}")
        print(f"  input: {block.input}")

## §3. 도구 정보 추출 (Extracting Tool Info)

ToolUseBlock에서 핵심 정보 3가지를 추출합니다:
- `id` — 이 특정 도구 호출의 고유 ID (결과 전달 시 필요)
- `name` — 어떤 도구를 호출할지
- `input` — 도구에 전달할 인자 (dict)

In [ ]:
# 도구 호출 블록 찾기
tool_use_block = None
for block in response.content:
    if block.type == "tool_use":
        tool_use_block = block
        break

if tool_use_block:
    tool_use_id = tool_use_block.id
    tool_name = tool_use_block.name
    tool_input = tool_use_block.input

    print(f"Tool Use ID: {tool_use_id}")
    print(f"Tool Name:   {tool_name}")
    print(f"Tool Input:  {tool_input}")

    # 실제 함수 실행
    result = get_current_datetime(**tool_input)
    print(f"\nExecution Result: {result}")
else:
    print("No tool_use block found.")

## §4. tool_result 메시지 구성 (Building Tool Result)

도구 실행 결과를 Claude에게 돌려보내려면:
1. Assistant의 응답 전체를 `messages`에 추가
2. `role: "user"`, `content`에 `tool_result` 블록을 담은 메시지 추가
3. `tool_use_id`가 반드시 **매칭**되어야 함

In [ ]:
# Step 1: assistant 응답을 대화에 추가
messages.append({"role": "assistant", "content": response.content})

# Step 2: tool_result 메시지 구성
tool_result_message = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_id,  # 반드시 매칭!
            "content": result,
        }
    ],
}

messages.append(tool_result_message)

print("Messages so far:")
for i, msg in enumerate(messages):
    print(f"  [{i}] role={msg['role']}, type={'tool_result' if isinstance(msg.get('content'), list) and msg['content'] and isinstance(msg['content'][0], dict) and msg['content'][0].get('type') == 'tool_result' else 'normal'}")

## §5. 최종 응답 수신 (Getting Final Response)

tool_result를 보내면 Claude가 결과를 해석하여 자연어 응답을 생성합니다.

In [ ]:
final_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=messages,
)

print("stop_reason:", final_response.stop_reason)
print()
for block in final_response.content:
    if block.type == "text":
        print("Claude:", block.text)

## §6. 에러 핸들링 (Error Handling with is_error)

도구 실행 중 에러가 발생하면, `is_error: True`로 Claude에게 알려줍니다.  
Claude는 에러 메시지를 보고 다른 방법을 시도하거나 사용자에게 안내합니다.

In [ ]:
# 일부러 잘못된 포맷으로 에러를 시뮬레이션
error_messages = [
    {"role": "user", "content": "What time is it?"}
]

error_response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=error_messages,
)

# assistant 응답 추가
error_messages.append({"role": "assistant", "content": error_response.content})

# 에러가 있는 tool_result 전송
for block in error_response.content:
    if block.type == "tool_use":
        error_tool_result = {
            "role": "user",
            "content": [
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": "Error: The datetime service is temporarily unavailable.",
                    "is_error": True,
                }
            ],
        }
        error_messages.append(error_tool_result)
        break

# Claude의 에러 대응 확인
error_final = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=error_messages,
)

print("Claude's error response:")
for block in error_final.content:
    if block.type == "text":
        print(block.text)